In [ ]:
import os
import json
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from hdbscan import HDBSCAN

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import seaborn as sns
import matplotlib.pyplot as plt

from utils.functions import load_yaml
from utils.graphfunction import load_graph, get_uniprot_from_nodes, get_res_from_nodes
from data.augmentation import ss_aug
from data.scaler import sin_cos_transform, pca_transform
from data.neighborExtractor import extract_neighbor_features, extract_neighbor_features_parallel

from data.vizutils import plot_cluster_distribution

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load Configuration
config = load_yaml("../config/RRGCL.yaml")
DATABASE = config.DATABASE

# Load Graph Data
finalG = load_graph((f"{DATABASE}/cleaned_weighted_graph_041226.pkl"))

# Feature DataFrame
org_res_feat_df = pd.read_csv(f'{config.DATABASE}/merged_feature_data_v041226.csv')
# org_ngb_feat_df = pd.read_csv('../data/proc_data/ngb_weighted_mean_feat.csv')
org_ngb_feat_df = pd.read_csv('../data/proc_data/ngb_orthogonal_proj_feat.csv')

# 1. Clustering

## 1-1 Data Ready (NaN Processing)

In [ ]:
ss_cols = ['rel_sasa', 'depth', 'hse_up', 'hse_down', 'dssp_accessibility', 'dssp_TCO', 'dssp_alpha', 'dssp_phi', 'dssp_psi']

In [ ]:
aa1_cols = [col for col in org_res_feat_df.columns if col.startswith('aa1')]
aa1_cols.remove('aa1')

In [ ]:
res_AA1 = org_res_feat_df[['node_id']+aa1_cols].copy()
ngb_AA1 = org_ngb_feat_df[['node_id']+aa1_cols].copy()

In [ ]:
org_res_feat_df.head(3)

In [ ]:
proc_res_feat_df = pd.merge(org_res_feat_df, res_AA1, on='node_id', how='right')
proc_res_feat_df

In [ ]:
cluster_cols = ['node_id',
                'aa1_KYTJ820101', 'aa1_KLEP840101', 'aa1_BHAR880101', 'aa1_JANJ780101', 'aa1_CHOP780201', 'aa1_GRAR740102', 'aa1_GRAR740103',
                'rel_sasa', 'depth', 'hse_up', 'hse_down', 'dssp_accessibility', 'dssp_TCO', 'dssp_alpha', 'dssp_phi', 'dssp_psi',
                ]

ngb_cluster_cols = cluster_cols.copy() + ['ss_helix', 'ss_sheet', 'ss_loop']

cluster_res_feat_df = org_res_feat_df[cluster_cols]
cluster_ngb_feat_df = org_ngb_feat_df[ngb_cluster_cols]

In [ ]:
cluster_ngb_feat_df

In [ ]:
aug_cluster_res_feat_df = ss_aug(cluster_res_feat_df, ss_cols, mode='neighbor', graph=finalG, num_ref=10)
aug_cluster_ngb_feat_df = ss_aug(cluster_ngb_feat_df, ss_cols+['ss_helix', 'ss_sheet', 'ss_loop'], mode='neighbor', graph=finalG, num_ref=10)

# Residue DataFrame
aug_cluster_res_feat_df = sin_cos_transform(aug_cluster_res_feat_df, 'dssp_phi', 'sin')
aug_cluster_res_feat_df = sin_cos_transform(aug_cluster_res_feat_df, 'dssp_phi', 'cos')

aug_cluster_res_feat_df = sin_cos_transform(aug_cluster_res_feat_df, 'dssp_psi', 'sin')
aug_cluster_res_feat_df = sin_cos_transform(aug_cluster_res_feat_df, 'dssp_psi', 'cos')

aug_cluster_res_feat_df = sin_cos_transform(aug_cluster_res_feat_df, 'dssp_alpha', 'sin')
aug_cluster_res_feat_df = sin_cos_transform(aug_cluster_res_feat_df, 'dssp_alpha', 'cos')

# Neighbor DataFrame
aug_cluster_ngb_feat_df = sin_cos_transform(aug_cluster_ngb_feat_df, 'dssp_phi', 'sin')
aug_cluster_ngb_feat_df = sin_cos_transform(aug_cluster_ngb_feat_df, 'dssp_phi', 'cos')

aug_cluster_ngb_feat_df = sin_cos_transform(aug_cluster_ngb_feat_df, 'dssp_psi', 'sin')
aug_cluster_ngb_feat_df = sin_cos_transform(aug_cluster_ngb_feat_df, 'dssp_psi', 'cos')

aug_cluster_ngb_feat_df = sin_cos_transform(aug_cluster_ngb_feat_df, 'dssp_alpha', 'sin')
aug_cluster_ngb_feat_df = sin_cos_transform(aug_cluster_ngb_feat_df, 'dssp_alpha', 'cos')

In [ ]:
corr = aug_cluster_res_feat_df.select_dtypes(include=['number']).corr()

plt.figure(figsize=(15, 12))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('[Residue] Features Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
corr = aug_cluster_ngb_feat_df.select_dtypes(include=['number']).corr()

plt.figure(figsize=(15, 12))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('[Neighbor] Features Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# aug_cluster_res_feat_df.to_csv('../data/proc_data/aug_res_feature_data_for_clustering.csv', index=False)
# aug_cluster_ngb_feat_df.to_csv('../data/proc_data/aug_ngb_orthogonal_feature_data_for_clustering.csv', index=False)

## 1-2 Feature Engineering for Clustering

### AAIndex1

In [ ]:
reS_scaler = StandardScaler()
ngb_scaler = StandardScaler()
res_AA1[aa1_cols] = reS_scaler.fit_transform(res_AA1[aa1_cols])
ngb_AA1[aa1_cols] = ngb_scaler.fit_transform(ngb_AA1[aa1_cols])

In [ ]:
res_pca_aa1_df, res_pca = pca_transform(res_AA1, aa1_cols, 2)
ngb_pca_aa1_df, ngb_pca = pca_transform(ngb_AA1, aa1_cols, 2)
print("[Residue] AAindex Explained variance:", res_pca.explained_variance_ratio_, res_pca.explained_variance_ratio_.sum())
print("[Neighbor] AAindex Explained variance:", ngb_pca.explained_variance_ratio_, ngb_pca.explained_variance_ratio_.sum())

In [ ]:
proc_cluster_res_feat_df = aug_cluster_res_feat_df.drop(aa1_cols, axis=1)
proc_cluster_ngb_feat_df = aug_cluster_ngb_feat_df.drop(aa1_cols, axis=1)

proc_cluster_res_feat_df = pd.merge(proc_cluster_res_feat_df, res_pca_aa1_df, on='node_id', how=ㅋ'left')
proc_cluster_ngb_feat_df = pd.merge(proc_cluster_ngb_feat_df, ngb_pca_aa1_df, on='node_id', how='left')

### Secondary Strcture

In [ ]:
ss_pca_cols1 = ['rel_sasa', 'depth', 'hse_up', 'hse_down', 'dssp_accessibility']
ss_pca_cols2 = ['dssp_TCO', 'dssp_phi_sin', 'dssp_phi_cos', 'dssp_psi_sin', 'dssp_psi_cos', 'dssp_alpha_sin', 'dssp_alpha_cos']

all_ss_pca_cols = ss_pca_cols1 + ss_pca_cols2

In [ ]:
reS_scaler = StandardScaler()
ngb_scaler = StandardScaler()
proc_cluster_res_feat_df[all_ss_pca_cols] = reS_scaler.fit_transform(proc_cluster_res_feat_df[all_ss_pca_cols])
proc_cluster_ngb_feat_df[all_ss_pca_cols] = ngb_scaler.fit_transform(proc_cluster_ngb_feat_df[all_ss_pca_cols])

In [ ]:
res_pca_ss_df, ss_res_pca = pca_transform(proc_cluster_res_feat_df, ss_pca_cols1, 2)
ngb_pca_ss_df, ss_ngb_pca = pca_transform(proc_cluster_ngb_feat_df, ss_pca_cols1, 2)
print("[Residue] ss1 Explained variance:", ss_res_pca.explained_variance_ratio_, ss_res_pca.explained_variance_ratio_.sum())
print("[Neighbor] ss1 Explained variance:", ss_ngb_pca.explained_variance_ratio_, ss_ngb_pca.explained_variance_ratio_.sum())

In [ ]:
res_pca_ss_df2, ss_res_pca2 = pca_transform(proc_cluster_res_feat_df, ss_pca_cols2, 2)
ngb_pca_ss_df2, ss_ngb_pca2 = pca_transform(proc_cluster_ngb_feat_df, ss_pca_cols2, 4)
print("[Residue] ss2 Explained variance:", ss_res_pca2.explained_variance_ratio_, ss_res_pca2.explained_variance_ratio_.sum())
print("[Neighbor] ss2 Explained variance:", ss_ngb_pca2.explained_variance_ratio_, ss_ngb_pca2.explained_variance_ratio_.sum())

In [ ]:
proc_cluster_res_feat_df = proc_cluster_res_feat_df.drop(all_ss_pca_cols+['dssp_alpha', 'dssp_phi', 'dssp_psi'], axis=1)
proc_cluster_ngb_feat_df = proc_cluster_ngb_feat_df.drop(all_ss_pca_cols+['dssp_alpha', 'dssp_phi', 'dssp_psi'], axis=1)

proc_cluster_res_feat_df = pd.merge(proc_cluster_res_feat_df, res_pca_ss_df, on='node_id', how='left')
proc_cluster_res_feat_df = pd.merge(proc_cluster_res_feat_df, res_pca_ss_df2, on='node_id', how='left')

proc_cluster_ngb_feat_df = pd.merge(proc_cluster_ngb_feat_df, ngb_pca_ss_df, on='node_id', how='left')
proc_cluster_ngb_feat_df = pd.merge(proc_cluster_ngb_feat_df, ngb_pca_ss_df2, on='node_id', how='left')

In [ ]:
corr = proc_cluster_res_feat_df.select_dtypes(include=['number']).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('[Residue] Proc Features Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
corr = proc_cluster_ngb_feat_df.select_dtypes(include=['number']).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('[Neighbor] Proc Features Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
data = proc_cluster_res_feat_df.select_dtypes(include=['number']).sample(5000, random_state=42)
scaled_data = StandardScaler().fit_transform(data)

tsne_params = {
    'n_components': 2,
    'perplexity': 30,
    'random_state': 42,
    'n_jobs': -1
}

projection = TSNE(**tsne_params).fit_transform(scaled_data)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=projection[:, 0], 
    y=projection[:, 1], 
    s=2,
    alpha=0.3,
    linewidth=0
)
plt.show()

In [ ]:
data = proc_cluster_ngb_feat_df.select_dtypes(include=['number']).sample(5000, random_state=42) # [['aa1_PCA0', 'aa1_PCA1', 'dssp_PCA0', 'dssp_PCA1', 'dssp_PCA2', 'rel_PCA0', 'rel_PCA1']]
scaled_data = StandardScaler().fit_transform(data)

tsne_params = {
    'n_components': 2,
    'perplexity': 30,
    'random_state': 42,
    'n_jobs': -1
}

projection = TSNE(**tsne_params).fit_transform(scaled_data)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=projection[:, 0], 
    y=projection[:, 1], 
    s=2,
    alpha=0.3,
    linewidth=0
)
plt.show()

## Cluster Analysis

In [ ]:
# res_cluster_results = pd.read_csv('')
# ngb_cluster_results = pd.read_csv('')

In [ ]:
# Add mapping features
proc_cluster_res_feat_df['residue'] = proc_cluster_res_feat_df['node_id'].map(get_res_from_nodes)
proc_cluster_res_feat_df['uniprot'] = proc_cluster_res_feat_df['node_id'].map(get_uniprot_from_nodes)

proc_cluster_ngb_feat_df['residue'] = proc_cluster_ngb_feat_df['node_id'].map(get_res_from_nodes)
proc_cluster_ngb_feat_df['uniprot'] = proc_cluster_ngb_feat_df['node_id'].map(get_uniprot_from_nodes)

In [ ]:
print("Residue-Level Clustering: Residue Types")
plot_cluster_distribution(proc_cluster_res_feat_df, 'residue', grid_rows=3, grid_cols=4)

print("Neighbor-Level Clustering: Residue Types")
plot_cluster_distribution(proc_cluster_ngb_feat_df, 'residue', grid_rows=3, grid_cols=4)

# 2. Similarity

In [ ]:
org_res_feat_df